# 01 — Exploratory Data Analysis
Arabic Aspect-Based Sentiment Analysis (ABSA) — DeepX Hackathon

Goals:
- Understand label distributions (aspects, sentiments)
- Analyse text properties (length, language mix, noise)
- Spot class imbalance early
- Visualise co-occurrence of aspects

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
import ast, re, warnings
warnings.filterwarnings('ignore')

# Arabic font support (optional — comment out if unavailable)
try:
    matplotlib.rcParams['font.family'] = 'DejaVu Sans'
except Exception:
    pass

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
TRAIN_PATH = 'data/DeepX_train.xlsx'
VAL_PATH   = 'data/DeepX_validation.xlsx'
UNLABELED_PATH = 'data/DeepX_unlabeled.xlsx'

train = pd.read_excel(TRAIN_PATH)
val   = pd.read_excel(VAL_PATH)
unlabeled = pd.read_excel(UNLABELED_PATH)

print(f'Train: {train.shape}  |  Val: {val.shape}  |  Unlabeled: {unlabeled.shape}')
train.head(3)

In [ ]:
# Parse stringified lists/dicts if needed
def safe_parse(x):
    if isinstance(x, list) or isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return x

for df in [train, val]:
    df['aspects'] = df['aspects'].apply(safe_parse)
    df['aspect_sentiments'] = df['aspect_sentiments'].apply(safe_parse)

labeled = pd.concat([train, val], ignore_index=True)
print('Combined labeled set:', labeled.shape)

## 2. Basic Stats

In [ ]:
print('=== Train ===')
print(train.dtypes)
print('\nMissing values:')
print(train.isnull().sum())
print('\nStar rating distribution:')
print(train['star_rating'].value_counts().sort_index())

In [ ]:
# Review text length (chars and words)
for df, name in [(train, 'Train'), (val, 'Val'), (unlabeled, 'Unlabeled')]:
    df['char_len'] = df['review_text'].astype(str).apply(len)
    df['word_len'] = df['review_text'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for df, name, color in [(train, 'Train', 'steelblue'), (val, 'Val', 'coral')]:
    axes[0].hist(df['char_len'].clip(upper=500), bins=40, alpha=0.6, label=name, color=color)
    axes[1].hist(df['word_len'].clip(upper=100), bins=40, alpha=0.6, label=name, color=color)
axes[0].set_title('Character Length Distribution'); axes[0].set_xlabel('Chars'); axes[0].legend()
axes[1].set_title('Word Length Distribution'); axes[1].set_xlabel('Words'); axes[1].legend()
plt.tight_layout(); plt.show()

print('Train word length — mean:', train['word_len'].mean().round(2), '| median:', train['word_len'].median())
print('Val  word length — mean:', val['word_len'].mean().round(2),   '| median:', val['word_len'].median())

## 3. Aspect Distribution

In [ ]:
VALID_ASPECTS = ['food','service','price','cleanliness','delivery','ambiance','app_experience','general','none']

all_aspects = [a for aspects in labeled['aspects'] for a in (aspects if isinstance(aspects, list) else [])]
aspect_counts = Counter(all_aspects)

fig, ax = plt.subplots(figsize=(10, 4))
keys = [k for k in VALID_ASPECTS if k in aspect_counts]
vals = [aspect_counts[k] for k in keys]
bars = ax.barh(keys, vals, color=sns.color_palette('muted', len(keys)))
ax.bar_label(bars, padding=3)
ax.set_title('Aspect Frequency (Train + Val)')
ax.set_xlabel('Count')
plt.tight_layout(); plt.show()

print('\nAspect counts:')
for k in VALID_ASPECTS:
    print(f'  {k:<18} {aspect_counts.get(k, 0)}')

## 4. Sentiment Distribution per Aspect

In [ ]:
rows = []
for _, row in labeled.iterrows():
    if isinstance(row['aspect_sentiments'], dict):
        for asp, sent in row['aspect_sentiments'].items():
            rows.append({'aspect': asp, 'sentiment': sent})
asp_sent_df = pd.DataFrame(rows)

pivot = asp_sent_df.groupby(['aspect','sentiment']).size().unstack(fill_value=0)
pivot = pivot.reindex(index=[a for a in VALID_ASPECTS if a in pivot.index])

pivot.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='white')
plt.title('Sentiment Breakdown per Aspect')
plt.xlabel('Aspect')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Sentiment')
plt.tight_layout(); plt.show()
print(pivot)

## 5. Multi-Label Aspect Count per Review

In [ ]:
labeled['n_aspects'] = labeled['aspects'].apply(lambda x: len(x) if isinstance(x, list) else 0)

fig, ax = plt.subplots(figsize=(8, 4))
labeled['n_aspects'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Number of Aspects per Review')
ax.set_xlabel('# Aspects')
ax.set_ylabel('Reviews')
plt.tight_layout(); plt.show()

print('Reviews with 0 aspects:', (labeled['n_aspects'] == 0).sum())
print('Reviews with 1 aspect: ', (labeled['n_aspects'] == 1).sum())
print('Reviews with 2+ aspects:', (labeled['n_aspects'] >= 2).sum())

## 6. Aspect Co-occurrence Heatmap

In [ ]:
from itertools import combinations

co = np.zeros((len(VALID_ASPECTS), len(VALID_ASPECTS)), dtype=int)
idx = {a: i for i, a in enumerate(VALID_ASPECTS)}

for aspects in labeled['aspects']:
    if isinstance(aspects, list):
        valid = [a for a in aspects if a in idx]
        for a, b in combinations(valid, 2):
            co[idx[a]][idx[b]] += 1
            co[idx[b]][idx[a]] += 1
        for a in valid:
            co[idx[a]][idx[a]] += 1

co_df = pd.DataFrame(co, index=VALID_ASPECTS, columns=VALID_ASPECTS)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(co_df, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Aspect Co-occurrence Matrix')
plt.tight_layout(); plt.show()

## 7. Business Category & Platform Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

train['business_category'].value_counts().head(15).plot(
    kind='barh', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Top Business Categories (Train)')

train['platform'].value_counts().plot(
    kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Platform Distribution (Train)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

## 8. Language & Noise Analysis

In [ ]:
def detect_script(text):
    text = str(text)
    arabic = len(re.findall(r'[\u0600-\u06FF]', text))
    latin  = len(re.findall(r'[a-zA-Z]', text))
    emoji  = len(re.findall(r'[\U00010000-\U0010ffff\U0001F600-\U0001F64F\U0001F300-\U0001F5FF]', text))
    total  = max(arabic + latin, 1)
    return {'arabic_ratio': arabic/total, 'latin_ratio': latin/total, 'has_emoji': int(emoji > 0)}

lang_stats = train['review_text'].apply(detect_script).apply(pd.Series)
train = pd.concat([train, lang_stats], axis=1)

print('Reviews with >50% Arabic:', (train['arabic_ratio'] > 0.5).sum())
print('Reviews with any Latin:  ', (train['latin_ratio'] > 0).sum())
print('Reviews with emoji:      ', train['has_emoji'].sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train['arabic_ratio'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Arabic Character Ratio')
axes[1].bar(['Has Emoji', 'No Emoji'], [train['has_emoji'].sum(), (~train['has_emoji'].astype(bool)).sum()],
            color=['gold', 'gray'])
axes[1].set_title('Emoji Presence')
plt.tight_layout(); plt.show()

## 9. Star Rating vs Sentiment Correlation

In [ ]:
sent_map = {'positive': 1, 'neutral': 0, 'negative': -1}

rating_sent = []
for _, row in train.iterrows():
    if isinstance(row['aspect_sentiments'], dict) and not pd.isna(row.get('star_rating')):
        avg_sent = np.mean([sent_map.get(s, 0) for s in row['aspect_sentiments'].values()])
        rating_sent.append({'star_rating': row['star_rating'], 'avg_sentiment': avg_sent})

rs_df = pd.DataFrame(rating_sent)
fig, ax = plt.subplots(figsize=(7, 4))
rs_df.groupby('star_rating')['avg_sentiment'].mean().plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('Average Sentiment Score by Star Rating')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Avg Sentiment (-1=neg, 0=neu, 1=pos)')
plt.tight_layout(); plt.show()

corr = rs_df.corr()['avg_sentiment']['star_rating']
print(f'Pearson correlation (star_rating vs avg_sentiment): {corr:.3f}')

## 10. EDA Summary & Key Takeaways

In [ ]:
print('=== EDA Summary ===')
print(f'Train samples        : {len(train)}')
print(f'Val samples          : {len(val)}')
print(f'Unlabeled samples    : {len(unlabeled)}')
print(f'Avg aspects/review   : {labeled["n_aspects"].mean():.2f}')
print(f'Most common aspect   : {aspect_counts.most_common(1)[0]}')
print(f'Avg review word len  : {labeled["word_len"].mean():.1f} words')
print()
print('Key observations:')
print('  1. Heavy class imbalance — service & food dominate')
print('  2. Most reviews are 1-2 aspects; rare 4+ aspect cases')
print('  3. High noise: emojis, code-switching (Arabic+Latin), slang')
print('  4. Star rating correlates well with average sentiment (use as auxiliary signal)')
print('  5. app_experience is platform-specific (play_store/app_store)')
print('  6. none aspect should only appear when NO real aspect is present')